# Resume a Run and Reuse Prior Work

Resume appends to one existing run; a new run may independently reuse eligible prior results. Neither operation claims an orphan worker seal or reruns unresolved science automatically.

## Prepare the checkout's canonical worker

Run this notebook with the checkout's Pixi `dev` kernel. The worker uses the same
`pyproject.toml` and committed `pixi.lock`, including the installed Artisan source
and declared development dependencies. Preparation captures bounded package
inputs; it does not copy `.git`, `.pixi`, or tutorial output directories.

The next cell prepares that worker if needed. First preparation may download
locked packages and needs disk space for a separate fixed-path environment.
Later operations and pipelines reuse it. Use the same source snapshot for the
controller and worker; changed package inputs require preparation again.
This setup does not provision models or GPUs. Submission only looks up an
existing prepared target.

`LocalRunner()` without a canonical environment merely observes the current
interpreter and is deliberately not cacheable. The notebook workspace is
temporary; use a persistent base directory when keeping runs.

In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
from tempfile import TemporaryDirectory

from artisan.operations.examples import DataGenerator, DataTransformer, MetricCalculator
from artisan.orchestration import PipelineManager, StepDisposition
from artisan.orchestration.runners import LocalRunner
from artisan.schemas import Environment
from artisan.utils import find_project_root, tutorial_setup
from artisan.visualization import inspect_pipeline

workspace = TemporaryDirectory(prefix="artisan-cache-tutorial-")
tutorial_setup = partial(tutorial_setup, base_dir=workspace.name)
env = tutorial_setup("cache")
project_root = find_project_root().resolve()
runner = LocalRunner(
    environment=Environment.from_pixi(project_root / "pyproject.toml", name="dev")
)
# These operations share the worker environment; prepare it once.
runner.prepare(DataGenerator)

In [ ]:
from artisan.operations.curator import Filter
from artisan.orchestration import list_runs
from artisan.schemas import CachePolicy

with PipelineManager.create(
    name="original",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
    preserve_staging=True,
) as original:
    original.run(DataGenerator, name="generate", params={"count": 2, "seed": 42})
    original.run(
        DataTransformer,
        name="transform",
        params={"seed": 100},
        inputs={"dataset": original.output("generate", "datasets")},
    )
    original.run(
        MetricCalculator,
        name="score",
        inputs={"dataset": original.output("transform", "dataset")},
    )
    original.run(
        Filter,
        name="selected",
        inputs={"passthrough": original.output("transform", "dataset")},
        params={
            "criteria": [
                {"metric": "distribution.median", "operator": "ge", "value": 0.0}
            ]
        },
    )
    assert original.finalize()["overall_success"]
original_run = original.config.pipeline_run_id
preserved = list(Path(env.staging_root).rglob("*.parquet"))
assert preserved
list_runs(env.delta_root)

## Resume the exact run

Pass the same explicit worker binding and run ID. Completed outcomes remain selected. `preserve_staging=True` keeps staged evidence available; resume does not reinterpret it as a new execution.

In [ ]:
with PipelineManager.resume(
    delta_root=env.delta_root,
    pipeline_run_id=original_run,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
    preserve_staging=True,
) as resumed:
    refined = resumed.run(
        DataTransformer,
        name="refine",
        params={"seed": 99},
        inputs={"dataset": resumed.output("selected", "passthrough")},
    )
    summary = resumed.finalize()
assert summary["overall_success"]
assert refined.step_number == 4
assert all(path.exists() for path in preserved)
inspect_pipeline(env.delta_root, pipeline_run_id=original_run)

## Start a new run using the same store

The same generation and transformation may reuse prior accepted results. A different Filter criterion is a different scientific configuration.

In [ ]:
with PipelineManager.create(
    name="new_run",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
) as later:
    source_hit = later.run(
        DataGenerator, name="generate", params={"count": 2, "seed": 42}
    )
    transform_hit = later.run(
        DataTransformer,
        name="transform",
        params={"seed": 100},
        inputs={"dataset": later.output("generate", "datasets")},
    )
    later.run(
        MetricCalculator,
        name="score",
        inputs={"dataset": later.output("transform", "dataset")},
    )
    changed_filter = later.run(
        Filter,
        name="selected",
        inputs={"passthrough": later.output("transform", "dataset")},
        params={
            "criteria": [
                {"metric": "distribution.median", "operator": "ge", "value": 0.5}
            ]
        },
    )
    assert later.finalize()["overall_success"]
assert source_hit.disposition is transform_hit.disposition is StepDisposition.CACHE_HIT
assert changed_filter.disposition is StepDisposition.EXECUTED
inspect_pipeline(env.delta_root, pipeline_run_id=later.config.pipeline_run_id)

## Select whole-step cache policy

`ALL_SUCCEEDED` requires a fully successful whole-step source; it may still reuse individual successful units while executing missing work. `STEP_COMPLETED` also permits an exact partial historical result, keeping its failures and accepted outputs. Failed, cancelled and diagnostic attempts are not whole-step cache sources.

In [ ]:
with PipelineManager.create(
    name="policy",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    default_step_runner=runner,
    cache_policy=CachePolicy.STEP_COMPLETED,
) as policy:
    selected = policy.run(
        DataGenerator,
        params={"count": 2, "seed": 42},
        cache_policy=CachePolicy.ALL_SUCCEEDED,
    )
    assert policy.finalize()["overall_success"]
assert selected.disposition is StepDisposition.CACHE_HIT